# agentorch 实验手册（完整功能版）

这个 notebook 已按当前最新的 `agentorch` v1 稳定公开面和 notebook 异步实验习惯重新整理，目标是把**当前已经支持的核心能力**都收口成可以直接实验的示例。

你可以把它当成：

1. facade-first 快速上手手册
2. notebook 异步实验脚本
3. API / 运行时 / 设计边界对齐清单

当前覆盖能力包括：

- OpenAI-compatible 模型接入与多媒体能力
- v1 稳定公开面：`create_agent(...)` / `create_multi_agent(...)`
- 设计入口：`AgentDesign` / `RoleDesign` / `TeamDesign` + `compose_*`
- 单 agent 基础运行
- 结构化工具注册与调用
- memory 与 thread 清理
- workflow 基础编排
- RAG-ready 检索接口与最小本地实现
- AgentRegistry 与 multi-agent delegation
- 联网搜索工具参数验证
- media tool 注入与多媒体模型示例
- elephant context 这类研究机制的组合边界


## 0. 先看最重要的使用规则

在 Jupyter / Notebook 里，现在有两层合法写法，需要分清：

1. **稳定公开面 / 日常优先**：`create_agent(...)`、`create_multi_agent(...)`、`AgentDesign` / `TeamDesign` + `compose_*`
2. **notebook 异步低层实验面**：`await Agent.acreate(...)`、`await Runtime.acreate(...)`、`await IndexedKnowledgeBase.acreate(...)`

这一份 notebook 的约定是：

- 能用 facade 讲清楚的地方，优先展示稳定公开面
- 需要验证 runtime 细粒度参数、显式 async constructor、底层组合边界时，再下沉到 `acreate(...)`
- 不要调用 `agent.run_sync(...)`
- 不要在 notebook 里调用 `Runtime.create(...)` / `Agent.create(...)` / `IndexedKnowledgeBase.create(...)`
- 涉及 tool calling、workflow、多 agent、memory、RAG 的实验时，建议每个示例使用独立 `thread_id`
- 如果修改了本地包代码，请重启 kernel 后重新运行

当前 notebook 与最新框架保持如下分层：

- `RuntimeConfig.agent(...)` / `RuntimeConfig.workflow(...)`
- `ContextPolicy / StatePolicy / CoordinationPolicy / MemoryPolicy`
- `AgentDesign / RoleDesign / TeamDesign` + `compose_agent(...)` / `compose_team(...)`
- `ToolRegistry.from_tools(...)` / `ToolRegistry.with_bundles(...)`
- `create_agent(...)` / `create_multi_agent(...)` 作为稳定公开面
- `experiments.elephant_context.build_elephant_runtime_config(...)` 这类研究机制保持插件式组合，不再作为主包预设代理

如果你只想先看 facade-first 示例和底层 runtime 示例如何分层，先看仓库里的 `examples/README.md`。


## 1. 环境准备

最低兼容版本：`Python 3.10+`

如果你和当前项目保持一致，优先使用现在这套环境：`data_analysis_py311`（Python 3.11）。

推荐安装命令：

```powershell
python -m pip install -e .
python -m pip install jupyterlab notebook
```

如果你要和命令行回归测试保持一致，也可以显式使用：

```powershell
C:/Users/24260/.conda/envs/data_analysis_py311/python.exe -m pip install -e .
```


In [ ]:
import sys
from pathlib import Path

print(sys.version)
print(Path.cwd())


## 2. API Key / Base URL 导入方式

这个 notebook 约定：`agentorch_experiments.ipynb` 和 `.env` 放在同一个目录下，并统一读取当前工作目录下的 `.env`。

说明：

- 核心库现在推荐 `OPENAI_*` 命名
- 但为了兼容你当前本地 `.env` 里已有的 `API_KEY / BASE_URL / APIYI_*` 等旧变量，这个 notebook 顶部会做一层本地兼容映射
- 映射后的值会显式传给 `ModelConfig` / `OpenAIModel` / `OpenAICompatibleHTTPModel`
- 这样既不破坏核心库的标准化语义，也能保证你现在这份 `.env` 直接可用

优先读取的变量大致分为两层：

1. 标准变量：
- `OPENAI_API_KEY` / `OPENAI_BASE_URL`
- `OPENAI_VISION_MODEL`
- `OPENAI_EMBEDDING_*`
- `OPENAI_TTS_*`
- `OPENAI_IMAGE_*`
- `OPENAI_VIDEO_*`

2. notebook 兼容层会额外识别你当前本地常见旧变量：
- `API_KEY` / `BASE_URL`
- `APIYI_KEY`
- `APIYI_IMAGE_BASEURL` / `APIYI_IMAGE_MODEL`
- 以及少量媒体相关的本地辅助字段

所以后面整本 notebook 不再假设公共默认供应商，而是都走这一份 `.env` 解析后的显式配置。


In [ ]:
import os
from pathlib import Path

from agentorch.config import ModelConfig, initialize_environment

NOTEBOOK_PROJECT_ROOT = Path.cwd()
NOTEBOOK_ENV_PATH = NOTEBOOK_PROJECT_ROOT / '.env'
initialize_environment(NOTEBOOK_ENV_PATH)


def load_env_file(env_path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not env_path.exists():
        return values
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


NOTEBOOK_ENV_VALUES = load_env_file(NOTEBOOK_ENV_PATH)


def pick_env(*names: str) -> str | None:
    for name in names:
        value = NOTEBOOK_ENV_VALUES.get(name)
        if value and value.strip():
            return value.strip()
        value = os.getenv(name)
        if value and value.strip():
            return value.strip()
    return None


def pick_int_env(*names: str) -> int | None:
    value = pick_env(*names)
    if value is None:
        return None
    try:
        return int(value)
    except ValueError:
        return None


def pick_float_env(*names: str, default: float | None = None) -> float | None:
    value = pick_env(*names)
    if value is None:
        return default
    try:
        return float(value)
    except ValueError:
        return default


def pick_csv_env(*names: str) -> list[str]:
    value = pick_env(*names)
    if not value:
        return []
    return [item.strip() for item in value.split(',') if item.strip()]


def resolve_demo_path(value: str | None, fallback_name: str) -> Path:
    raw = value.strip() if value and value.strip() else fallback_name
    candidate = Path(raw).expanduser()
    if not candidate.is_absolute():
        candidate = NOTEBOOK_PROJECT_ROOT / candidate
    return candidate


NOTEBOOK_CHAT_API_KEY = pick_env('OPENAI_API_KEY', 'API_KEY', 'APIYI_KEY')
NOTEBOOK_CHAT_BASE_URL = pick_env('OPENAI_BASE_URL', 'BASE_URL')
NOTEBOOK_CHAT_MODEL = pick_env('OPENAI_MODEL', 'OPENAI_CHAT_MODEL', 'AGENTORCH_CHAT_MODEL') or 'gpt-4.1-mini'
NOTEBOOK_VISION_MODEL = pick_env('OPENAI_VISION_MODEL')

NOTEBOOK_EMBEDDING_API_KEY = pick_env('OPENAI_EMBEDDING_API_KEY', 'OPENAI_API_KEY', 'API_KEY', 'APIYI_KEY')
NOTEBOOK_EMBEDDING_BASE_URL = pick_env('OPENAI_EMBEDDING_BASE_URL', 'OPENAI_BASE_URL', 'BASE_URL')
NOTEBOOK_EMBEDDING_MODEL = pick_env('OPENAI_EMBEDDING_MODEL') or 'text-embedding-3-small'
NOTEBOOK_EMBEDDING_DIMENSIONS = pick_int_env('OPENAI_EMBEDDING_DIMENSIONS')

NOTEBOOK_SPEECH_API_KEY = pick_env('OPENAI_TTS_API_KEY', 'OPENAI_API_KEY', 'API_KEY', 'APIYI_KEY')
NOTEBOOK_SPEECH_BASE_URL = pick_env('OPENAI_TTS_BASE_URL', 'OPENAI_BASE_URL', 'BASE_URL')
NOTEBOOK_SPEECH_MODEL = pick_env('OPENAI_TTS_MODEL')
NOTEBOOK_SPEECH_VOICE = pick_env('OPENAI_TTS_VOICE')
NOTEBOOK_SPEECH_FORMAT = pick_env('OPENAI_TTS_FORMAT') or 'mp3'
NOTEBOOK_SPEECH_SPEED = pick_float_env('OPENAI_TTS_SPEED', default=1.0)

NOTEBOOK_IMAGE_API_KEY = pick_env('OPENAI_IMAGE_API_KEY', 'APIYI_IMAGE_API_KEY', 'APIYI_KEY_image', 'APIYI_KEY', 'OPENAI_API_KEY', 'API_KEY')
NOTEBOOK_IMAGE_BASE_URL = pick_env('OPENAI_IMAGE_BASE_URL', 'APIYI_IMAGE_BASEURL', 'APIYI_IMAGE_BASE_URL')
NOTEBOOK_IMAGE_EXPLICIT_URL = pick_env('OPENAI_IMAGE_EXPLICIT_URL', 'OPENAI_IMAGE_URL', 'APIYI_GEN_URL')
NOTEBOOK_IMAGE_MODEL = pick_env('OPENAI_IMAGE_MODEL', 'APIYI_IMAGE_MODEL')
NOTEBOOK_IMAGE_ASPECT_RATIO = pick_env('OPENAI_IMAGE_ASPECT_RATIO', 'APIYI_GEN_ASPECT_RATIO') or '16:9'
NOTEBOOK_IMAGE_SIZE = pick_env('OPENAI_IMAGE_SIZE', 'APIYI_GEN_IMAGE_SIZE') or '2K'
NOTEBOOK_IMAGE_TIMEOUT = pick_float_env('OPENAI_IMAGE_TIMEOUT', 'APIYI_GEN_TIMEOUT', default=300.0)
NOTEBOOK_IMAGE_FALLBACK_MODELS = pick_csv_env('OPENAI_IMAGE_FALLBACK_MODELS', 'APIYI_GEN_FALLBACK_MODELS')

NOTEBOOK_VIDEO_API_KEY = pick_env('OPENAI_VIDEO_API_KEY', 'APIYI_KEY_ROLE', 'APIYI_KEY', 'OPENAI_API_KEY', 'API_KEY')
NOTEBOOK_VIDEO_BASE_URL = pick_env('OPENAI_VIDEO_BASE_URL', 'APIYI_VIDEO_BASE_URL', 'OPENAI_BASE_URL', 'BASE_URL')
NOTEBOOK_VIDEO_MODEL = pick_env('OPENAI_VIDEO_MODEL', 'APIYI_VIDEO_MODEL')

NOTEBOOK_IMAGE_DEMO_PATH = resolve_demo_path(pick_env('NOTEBOOK_IMAGE_PATH', 'APIYI_IMAGE_PATH'), 'your_image.png')
NOTEBOOK_VIDEO_DEMO_PATH = resolve_demo_path(pick_env('NOTEBOOK_VIDEO_PATH', 'APIYI_VIDEO_PATH'), 'your_video.mp4')
NOTEBOOK_IMAGE_GENERATION_PROMPT = pick_env('NOTEBOOK_IMAGE_PROMPT', 'APIYI_IMAGE_PROMPT') or 'A clean research-notebook illustration about multi-agent orchestration and embeddings.'

# 兼容旧版 notebook 单元中遗留的变量名，避免刷新不完整时出现 NameError。
DEFAULT_EMBEDDING_MODEL = NOTEBOOK_EMBEDDING_MODEL
DEFAULT_SPEECH_MODEL = NOTEBOOK_SPEECH_MODEL
DEFAULT_SPEECH_VOICE = NOTEBOOK_SPEECH_VOICE
DEFAULT_IMAGE_MODEL = NOTEBOOK_IMAGE_MODEL
DEFAULT_VIDEO_MODEL = NOTEBOOK_VIDEO_MODEL
IMAGE_DEMO_PATH = NOTEBOOK_IMAGE_DEMO_PATH
VIDEO_DEMO_PATH = NOTEBOOK_VIDEO_DEMO_PATH

cfg = ModelConfig.from_any(
    NOTEBOOK_CHAT_MODEL,
    api_key=NOTEBOOK_CHAT_API_KEY,
    base_url=NOTEBOOK_CHAT_BASE_URL,
    temperature=0,
    vision_model=NOTEBOOK_VISION_MODEL,
    embedding_api_key=NOTEBOOK_EMBEDDING_API_KEY,
    embedding_base_url=NOTEBOOK_EMBEDDING_BASE_URL,
    embedding_model=NOTEBOOK_EMBEDDING_MODEL,
    embedding_dimensions=NOTEBOOK_EMBEDDING_DIMENSIONS,
    speech_api_key=NOTEBOOK_SPEECH_API_KEY,
    speech_base_url=NOTEBOOK_SPEECH_BASE_URL,
    speech_model=NOTEBOOK_SPEECH_MODEL,
    speech_voice=NOTEBOOK_SPEECH_VOICE,
    speech_format=NOTEBOOK_SPEECH_FORMAT,
    speech_speed=NOTEBOOK_SPEECH_SPEED,
    image_api_key=NOTEBOOK_IMAGE_API_KEY,
    image_base_url=NOTEBOOK_IMAGE_BASE_URL,
    image_explicit_url=NOTEBOOK_IMAGE_EXPLICIT_URL,
    image_model=NOTEBOOK_IMAGE_MODEL,
    image_aspect_ratio=NOTEBOOK_IMAGE_ASPECT_RATIO,
    image_size=NOTEBOOK_IMAGE_SIZE,
    image_timeout=NOTEBOOK_IMAGE_TIMEOUT,
    image_fallback_models=NOTEBOOK_IMAGE_FALLBACK_MODELS,
    video_api_key=NOTEBOOK_VIDEO_API_KEY,
    video_base_url=NOTEBOOK_VIDEO_BASE_URL,
    video_model=NOTEBOOK_VIDEO_MODEL,
)

print('.env exists:', NOTEBOOK_ENV_PATH.exists())
print('legacy keys present:', [key for key in ['API_KEY', 'BASE_URL', 'APIYI_KEY', 'APIYI_IMAGE_BASEURL', 'APIYI_IMAGE_MODEL'] if key in NOTEBOOK_ENV_VALUES])
print('chat api_key loaded:', bool(NOTEBOOK_CHAT_API_KEY))
print('chat base_url loaded:', bool(cfg.base_url))
print('chat model:', cfg.model)
print('vision model:', cfg.vision_model)
print('embedding model:', cfg.embedding_model)
print('speech model:', cfg.speech_model)
print('speech voice:', cfg.speech_voice)
print('image model:', cfg.image_model)
print('video model:', cfg.video_model)
print('image demo file name:', NOTEBOOK_IMAGE_DEMO_PATH.name)
print('video demo file name:', NOTEBOOK_VIDEO_DEMO_PATH.name)
print('timeout:', cfg.timeout)


## 3. 当前公开 API 与策略入口

这一节建议重点关注四类入口：

- 顶层稳定公开面：`create_agent(...)` / `create_multi_agent(...)`
- 设计入口：`AgentDesign / RoleDesign / TeamDesign` + `compose_agent(...)` / `compose_team(...)`
- 统一配置入口：`RuntimeConfig.agent(...)` / `RuntimeConfig.workflow(...)`
- 可选策略与研究插件：`ContextPolicy / StatePolicy / CoordinationPolicy / MemoryPolicy`；象群实验入口：`experiments.elephant_context.build_elephant_runtime_config()`


In [ ]:
import agentorch
from agentorch import ContextPolicy, CoordinationPolicy, MemoryPolicy, StatePolicy
from experiments.elephant_context import build_elephant_runtime_config

print('public api sample:', agentorch.__all__[:40], '...')
print('default context policy:', ContextPolicy.default().model_dump())
print('default state policy:', StatePolicy().model_dump())
print('default coordination policy:', CoordinationPolicy().model_dump())
print('default memory policy:', MemoryPolicy().model_dump())
print('elephant plugin route mode:', build_elephant_runtime_config().coordination_policy.route_mode)


## 3.1 模型能力调用示例

当前 `agentorch` 已把模型能力统一挂到 `OpenAIModel` / `OpenAICompatibleHTTPModel` 上。你可以直接调用：

- `generate(...)`：标准聊天/推理
- `analyze_image(...)`：图片理解
- `embed(...)` / `embed_text(...)`：embedding
- `synthesize_speech(...)`：文本转语音
- `generate_image(...)`：图片生成
- `analyze_video(...)`：视频理解

这一节会**统一复用第 2 节从 `.env` 解析出的显式配置**。也就是说：

- 核心库仍然保持标准化，不再默认读取 `API_KEY / BASE_URL / APIYI_*`
- 但这个 notebook 会把你当前 `.env` 里的旧变量映射到显式 `ModelConfig` 字段
- 所以后面所有能力示例都能直接吃这份 `.env`

为了避免一打开 notebook 就消耗额度，所有真实调用默认都用 `RUN_* = False` 关闭，需要你手动打开。


In [ ]:
from agentorch import OpenAICompatibleHTTPModel, OpenAIModel, ToolRegistry
from agentorch.config import ModelConfig
from agentorch.core import Message, ModelRequest

NOTEBOOK_CAPABILITY_OUTPUT_DIR = NOTEBOOK_PROJECT_ROOT / '.agentorch' / 'notebook_capability_examples'
NOTEBOOK_CAPABILITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def make_notebook_model(
    *,
    provider: str = 'sdk',
    chat_model: str = NOTEBOOK_CHAT_MODEL,
    vision_model: str | None = NOTEBOOK_VISION_MODEL,
    embedding_model: str | None = NOTEBOOK_EMBEDDING_MODEL,
    speech_model: str | None = NOTEBOOK_SPEECH_MODEL,
    speech_voice: str | None = NOTEBOOK_SPEECH_VOICE,
    image_model: str | None = NOTEBOOK_IMAGE_MODEL,
    video_model: str | None = NOTEBOOK_VIDEO_MODEL,
):
    config = ModelConfig.from_any(
        chat_model,
        api_key=NOTEBOOK_CHAT_API_KEY,
        base_url=NOTEBOOK_CHAT_BASE_URL,
        temperature=0,
        vision_model=vision_model,
        embedding_api_key=NOTEBOOK_EMBEDDING_API_KEY,
        embedding_base_url=NOTEBOOK_EMBEDDING_BASE_URL,
        embedding_model=embedding_model,
        embedding_dimensions=NOTEBOOK_EMBEDDING_DIMENSIONS,
        speech_api_key=NOTEBOOK_SPEECH_API_KEY,
        speech_base_url=NOTEBOOK_SPEECH_BASE_URL,
        speech_model=speech_model,
        speech_voice=speech_voice,
        speech_format=NOTEBOOK_SPEECH_FORMAT,
        speech_speed=NOTEBOOK_SPEECH_SPEED,
        image_api_key=NOTEBOOK_IMAGE_API_KEY,
        image_base_url=NOTEBOOK_IMAGE_BASE_URL,
        image_explicit_url=NOTEBOOK_IMAGE_EXPLICIT_URL,
        image_model=image_model,
        image_aspect_ratio=NOTEBOOK_IMAGE_ASPECT_RATIO,
        image_size=NOTEBOOK_IMAGE_SIZE,
        image_timeout=NOTEBOOK_IMAGE_TIMEOUT,
        image_fallback_models=NOTEBOOK_IMAGE_FALLBACK_MODELS,
        video_api_key=NOTEBOOK_VIDEO_API_KEY,
        video_base_url=NOTEBOOK_VIDEO_BASE_URL,
        video_model=video_model,
    )
    if provider == 'http':
        return OpenAICompatibleHTTPModel.from_config(config)
    return OpenAIModel.from_config(config)


print('.env exists   ->', NOTEBOOK_ENV_PATH.exists())
print('chat model    ->', NOTEBOOK_CHAT_MODEL)
print('vision model  ->', NOTEBOOK_VISION_MODEL)
print('embedding     ->', NOTEBOOK_EMBEDDING_MODEL)
print('speech model  ->', NOTEBOOK_SPEECH_MODEL)
print('speech voice  ->', NOTEBOOK_SPEECH_VOICE)
print('image model   ->', NOTEBOOK_IMAGE_MODEL)
print('video model   ->', NOTEBOOK_VIDEO_MODEL)
print('image file    ->', NOTEBOOK_IMAGE_DEMO_PATH.name)
print('video file    ->', NOTEBOOK_VIDEO_DEMO_PATH.name)


### 3.1.1 直接聊天调用

如果你想绕过 `Agent`，直接测试底层模型适配器，可以直接构造 `ModelRequest`。这适合确认 `model / api_key / base_url` 是否已经配通。


In [ ]:
RUN_CHAT_MODEL_DEMO = False

if RUN_CHAT_MODEL_DEMO:
    model = make_notebook_model()
    try:
        request = ModelRequest(
            messages=[Message(role='user', content='请用两句话介绍 agentorch。')],
            max_tokens=128,
            temperature=0,
        )
        response = await model.generate(request)
        print(response.content)
    finally:
        await model.aclose()
else:
    print('Set RUN_CHAT_MODEL_DEMO = True before running this cell.')


### 3.1.2 图片理解 `analyze_image(...)`

如果图片理解和聊天共用同一个多模态模型，可以只保留 `chat_model`；如果图片理解要走单独模型，就在 `.env` 里配置 `OPENAI_VISION_MODEL`，或者在 `make_notebook_model(vision_model='...')` 里显式传入。


In [ ]:
RUN_IMAGE_UNDERSTANDING_DEMO = False

if RUN_IMAGE_UNDERSTANDING_DEMO:
    assert NOTEBOOK_IMAGE_DEMO_PATH.exists(), f'请先准备图片文件: {NOTEBOOK_IMAGE_DEMO_PATH.name}'
    model = make_notebook_model()
    try:
        response = await model.analyze_image(
            prompt='请描述这张图片中的主体、场景和最重要的细节。',
            image_path=NOTEBOOK_IMAGE_DEMO_PATH,
            max_tokens=256,
            temperature=0,
        )
        print(response.content)
    finally:
        await model.aclose()
else:
    print('Set RUN_IMAGE_UNDERSTANDING_DEMO = True and prepare NOTEBOOK_IMAGE_PATH in the local .env file or current directory.')


### 3.1.3 Embedding `embed_text(...)` / `embed(...)`

这是最接近你之前单独 `OpenAI().embeddings.create(...)` 的 notebook 版写法。这里直接通过 `agentorch` 的模型能力调用，不需要手写裸 SDK client。


In [ ]:
RUN_EMBEDDING_DEMO = False

if RUN_EMBEDDING_DEMO:
    model = make_notebook_model(embedding_model=NOTEBOOK_EMBEDDING_MODEL)
    try:
        text = '人工智能正在改变世界'
        vector = await model.embed_text(text)
        print('single vector dim:', len(vector))
        print('single vector first 5:', vector[:5])

        batch_vectors = await model.embed([
            '人工智能正在改变世界',
            '多智能体系统强调协作与分工',
            'embedding 可用于检索、聚类与召回',
        ])
        print('batch size:', len(batch_vectors))
        print('batch[0] dim:', len(batch_vectors[0]))
    finally:
        await model.aclose()
else:
    print('Set RUN_EMBEDDING_DEMO = True before running this cell.')


### 3.1.4 文本转语音 `synthesize_speech(...)`

TTS 需要 `speech_model` 和 `speech_voice`。它们不再由库默认选择，所以建议在 `.env` 里配置 `OPENAI_TTS_MODEL` / `OPENAI_TTS_VOICE`，或者在这里显式传入。


In [ ]:
RUN_TTS_DEMO = False

if RUN_TTS_DEMO:
    model = make_notebook_model(
        speech_model=NOTEBOOK_SPEECH_MODEL,
        speech_voice=NOTEBOOK_SPEECH_VOICE,
    )
    try:
        result = await model.synthesize_speech(
            '你好，这里是 agentorch notebook 中的文本转语音示例。',
            output_path=NOTEBOOK_CAPABILITY_OUTPUT_DIR / 'hello_tts',
        )
        print('audio path:', result.output_path)
        print('response format:', result.response_format)
        print('content type:', result.content_type)
        print('bytes written:', result.bytes_written)
    finally:
        await model.aclose()
else:
    print('Set RUN_TTS_DEMO = True and configure OPENAI_TTS_MODEL / OPENAI_TTS_VOICE first.')


### 3.1.5 图片生成 `generate_image(...)`

图片生成通常需要单独的 `OPENAI_IMAGE_*` 配置。你也可以改成 `provider='http'` 来测试 OpenAI-compatible HTTP 网关。


In [ ]:
RUN_IMAGE_GENERATION_DEMO = False

if RUN_IMAGE_GENERATION_DEMO:
    model = make_notebook_model(image_model=NOTEBOOK_IMAGE_MODEL)
    try:
        result = await model.generate_image(
            NOTEBOOK_IMAGE_GENERATION_PROMPT,
            output_path=NOTEBOOK_CAPABILITY_OUTPUT_DIR / 'hello_image',
        )
        print('image path:', result.output_path)
        print('mime type:', result.mime_type)
        print('bytes written:', result.bytes_written)
        print('model used:', result.model)
    finally:
        await model.aclose()
else:
    print('Set RUN_IMAGE_GENERATION_DEMO = True and configure OPENAI_IMAGE_MODEL first.')


### 3.1.6 视频理解 `analyze_video(...)`

视频理解会把本地视频转成 data URL 后发给模型，所以最适合先用较短的视频片段做实验。


In [ ]:
RUN_VIDEO_UNDERSTANDING_DEMO = False

if RUN_VIDEO_UNDERSTANDING_DEMO:
    assert NOTEBOOK_VIDEO_DEMO_PATH.exists(), f'请先准备视频文件: {NOTEBOOK_VIDEO_DEMO_PATH.name}'
    model = make_notebook_model(video_model=NOTEBOOK_VIDEO_MODEL)
    try:
        response = await model.analyze_video(
            prompt='请用三句话总结这个视频的核心内容，并指出最关键的动作或场景。',
            video_path=NOTEBOOK_VIDEO_DEMO_PATH,
            max_tokens=256,
            temperature=0,
        )
        print(response.content)
    finally:
        await model.aclose()
else:
    print('Set RUN_VIDEO_UNDERSTANDING_DEMO = True and prepare NOTEBOOK_VIDEO_PATH in the local .env file or current directory.')


### 3.1.7 `include_media=True` 自动注入 media tools

如果你后面要把这些能力挂到 agent 上，最方便的方式仍然是通过 `ToolRegistry.with_bundles(..., include_media=True)` 自动注入 `text_to_speech / generate_image / analyze_video`。


In [ ]:
media_model = make_notebook_model()
try:
    media_tools = ToolRegistry.with_bundles(
        workspace_root=NOTEBOOK_PROJECT_ROOT,
        include_filesystem=False,
        include_execution=False,
        include_git=False,
        include_media=True,
        model=media_model,
    )
    print('registered media tools:', [spec['function']['name'] for spec in media_tools.list_specs()])
finally:
    await media_model.aclose()


## 4. 最小 Agent 运行

这里优先用当前 v1 稳定公开面的 `create_agent(...)` 做 notebook 里的最小演示：

- 构造阶段走 facade
- 运行阶段仍然用 `await agent.run(...)`
- `RuntimeConfig.agent(...)` 统一挂载 `reasoning / prompt / strategy`
- 如果你后面要试 runtime 细粒度参数，再下沉到 `Agent.acreate(...)` / `Runtime.acreate(...)`
- 象群等研究范式通过 `experiments.elephant_context` 插件装配，不再进入核心 runtime 预设

下面是 notebook 环境下最小且和当前稳定公开面一致的写法。


In [ ]:
from agentorch import ContextPolicy, create_agent
from agentorch.config import RuntimeConfig

agent = create_agent(
    model=make_notebook_model(),
    name='nb-basic-agent',
    runtime_config=RuntimeConfig.agent(
        system_prompt='你是一个清晰、准确、简洁的 agentorch 助手。',
        context_policy=ContextPolicy.lean(),
        reasoning='react',
    ),
)

result = await agent.run(
    '请用三句话介绍 agentorch 这个包的作用。',
    thread_id='nb-basic-001',
)

print(result.output_text)
print('facade:', agent.export_blueprint()['facade'])
print('reasoning kind:', result.reasoning_kind)
print('resolved policies:', result.reasoning_metadata.get('resolved_policies'))
print('context budget:', result.reasoning_metadata.get('context_budget_report'))


## 5. 结构化工具实验

推荐的新写法：

- 用 `ToolRegistry.from_tools(...)` 一次性创建工具注册表
- 再通过 `create_agent(...)` 把 `tools` 装入 runtime
- notebook 里运行时仍然保持 async 风格：`await agent.run(...)`
- 如果后面要验证显式 async constructor，再切换到 `Agent.acreate(...)`


In [ ]:
from pydantic import BaseModel
from agentorch import ToolRegistry, create_agent, tool
from agentorch.config import RuntimeConfig


class AddInput(BaseModel):
    a: int
    b: int


@tool(description='Add two integers together.')
async def add_numbers(input: AddInput):
    return {'sum': input.a + input.b}


tools = ToolRegistry.from_tools(add_numbers)

agent = create_agent(
    model=make_notebook_model(),
    tools=tools,
    runtime_config=RuntimeConfig.agent(reasoning='react'),
    name='nb-tool-agent',
)

result = await agent.run(
    '请调用 add_numbers 工具，计算 123 + 456，并解释结果。',
    thread_id='nb-tool-003',
)

print(result.output_text)
print(result.tool_results)


### 如果你想复用旧 thread_id

先清理旧线程消息，避免旧状态残留：


In [ ]:
# 示例：
# await runtime.memory.clear_thread('nb-tool-001')


## 6. 查看工具调用的结构化结果


In [ ]:
for item in result.tool_results:
    print(item.model_dump())


### 6.1 一次回复触发多个工具调用

如果模型在同一次回复里规划出多个 `tool_calls`，`agentorch` 会把这些调用的执行结果统一收集到 `result.tool_results` 里。

下面这个例子要求模型在**同一次工具规划**里同时调用 `add_numbers` 和 `multiply_numbers`，便于观察多工具调用的结果聚合。


In [ ]:
from pydantic import BaseModel
from agentorch import ToolRegistry, create_agent, tool
from agentorch.config import RuntimeConfig


class MultiplyInput(BaseModel):
    a: int
    b: int


@tool(description='Multiply two integers together.')
async def multiply_numbers(input: MultiplyInput):
    return {'product': input.a * input.b}


multi_tools = ToolRegistry.from_tools(add_numbers, multiply_numbers)

multi_agent = create_agent(
    model=make_notebook_model(),
    tools=multi_tools,
    runtime_config=RuntimeConfig.agent(reasoning='react'),
    name='nb-multi-tool-agent',
)

multi_result = await multi_agent.run(
    '请严格按要求执行：在同一次回复里同时调用 add_numbers 和 multiply_numbers 两个工具，不要分两轮。先计算 12 + 34，再计算 12 * 34，最后用一句话总结两个结果。',
    thread_id='nb-tool-multi-001',
)

print(multi_result.output_text)
print('tool call count:', len(multi_result.tool_results))
print('tool names:', [item.tool_name for item in multi_result.tool_results])

for item in multi_result.tool_results:
    print(item.model_dump())


## 7. 多个工具调用稳定实验

如果你当前主要想验证“模型能不能在一次任务里调用多个工具”，最稳妥的方式是先不用 `python_interpreter`，而是改用几个纯 Python 函数工具。

这样可以避开本地沙箱、命令白名单、解释器路径等环境问题，把测试重点放在 `tool_calls` 和 `tool_results` 本身。


In [ ]:
from pydantic import BaseModel
from agentorch import ToolRegistry, create_agent, tool
from agentorch.config import RuntimeConfig


class AddInput(BaseModel):
    a: int
    b: int


@tool(description='Add two integers together.')
async def add_numbers(input: AddInput):
    return {'sum': input.a + input.b}


class MultiplyInput(BaseModel):
    a: int
    b: int


@tool(description='Multiply two integers together.')
async def multiply_numbers(input: MultiplyInput):
    return {'product': input.a * input.b}


class WeatherInput(BaseModel):
    city: str


@tool(description='Return a fake weather summary for demo/testing.')
async def get_weather(input: WeatherInput):
    mocked = {
        '上海': {'weather': '多云', 'temperature_c': 24},
        '北京': {'weather': '晴', 'temperature_c': 26},
        '深圳': {'weather': '小雨', 'temperature_c': 28},
    }
    return {'city': input.city, **mocked.get(input.city, {'weather': '未知', 'temperature_c': None})}


class CurrencyInput(BaseModel):
    amount_cny: float
    rate: float = 7.2


@tool(description='Convert CNY to USD with a mocked fixed exchange rate for demo/testing.')
async def convert_cny_to_usd(input: CurrencyInput):
    usd = round(input.amount_cny / input.rate, 2)
    return {'amount_cny': input.amount_cny, 'rate': input.rate, 'amount_usd': usd}


tools = ToolRegistry.from_tools(add_numbers, multiply_numbers, get_weather, convert_cny_to_usd)

agent = create_agent(
    model=make_notebook_model(),
    tools=tools,
    runtime_config=RuntimeConfig.agent(reasoning='react'),
    name='nb-stability-agent',
)

result = await agent.run(
    '请在同一次回复中同时调用 3 个工具，不要分多轮：1) 调用 add_numbers 计算 25 + 17；2) 调用 multiply_numbers 计算 8 * 9；3) 调用 get_weather 查询银川5月1日的天气。最后把三个工具结果整理成三行输出。',
    thread_id='nb-multi-tools-002',
)

print(result.output_text)
print('tool call count:', len(result.tool_results))
print('tool names:', [item.tool_name for item in result.tool_results])
for item in result.tool_results:
    print(item.model_dump())


## 8. 再做一次 4 工具联合调用压测


In [ ]:
stress_result = await agent.run(
    '请在同一次回复中连续调用 4 个工具，不要拆成多轮：1) add_numbers 计算 101 + 99；2) multiply_numbers 计算 7 * 11；3) get_weather 查询北京天气；4) convert_cny_to_usd 把 144 元人民币按默认汇率换算成美元。最后输出一个简短汇总。',
    thread_id='nb-multi-tools-003',
)

print(stress_result.output_text)
print('tool call count:', len(stress_result.tool_results))
print('tool names:', [item.tool_name for item in stress_result.tool_results])

for item in stress_result.tool_results:
    print(item.model_dump())


## 9. Memory 实验

当前 memory 主要支持：

- thread message
- thread summary
- long-term record
- checkpoint
- clear_thread


In [ ]:
from agentorch.memory import MemoryManager, MemoryRecord
from agentorch.core import Message

memory = MemoryManager()

await memory.append_message('thread-demo', Message(role='user', content='我偏好异步优先架构'))
await memory.append_message('thread-demo', Message(role='assistant', content='收到，我会按异步优先来设计。'))

summary = await memory.summarize_thread('thread-demo')
print(summary)

await memory.remember(
    MemoryRecord(
        thread_id='thread-demo',
        kind='preference',
        content='用户偏好异步优先架构',
        tags=['preference', 'architecture'],
    )
)

records = await memory.search(thread_id='thread-demo', query='异步')
records


## 10. 清理线程上下文


In [ ]:
await memory.clear_thread('thread-demo')
print(await memory.get_thread_messages('thread-demo'))


## 11. Workflow 基础实验

当前 workflow 除了基础节点，还已经支持更偏 RAG 编排的节点：

- `model`
- `tool`
- `router`
- `memory`
- `agent`
- `retrieve`
- `rag_router`
- `rag_mount`
- `rag_evaluate`

并且 workflow 节点可以局部覆盖：

- `rag_strategy` / `rag_mode`
- `context_policy`
- `state_policy`
- `coordination_policy`
- `memory_policy`

先从最基础的 memory + model 流程开始。


In [ ]:
from agentorch import WorkflowBuilder, create_agent
from agentorch.config import RuntimeConfig
from agentorch.workflow import Node

workflow = (
    WorkflowBuilder()
    .then(
        Node(
            id='remember_project',
            kind='memory',
            config={
                'action': 'remember',
                'kind': 'project_note',
                'content': '`agentorch` 是一个代码优先、异步优先的 Python 智能体编排框架，用来构建可编程的 agent 系统。它提供结构化工具、工作流、RAG、记忆、推理策略、沙箱执行和多智能体委派能力。',
                'tags': ['project', 'overview'],
            },
        )
    )
    .then(Node.model_node('summarize', prompt='请总结 agentorch 的定位和适合的使用场景。'))
    .build()
)

agent = create_agent(
    model=make_notebook_model(),
    workflow=workflow,
    runtime_config=RuntimeConfig.workflow(reasoning='react'),
    name='nb-workflow-basic-agent',
)

result = await agent.run('请开始执行这个 workflow。', thread_id='nb-workflow-basic-001')
print(result.output_text)


## 12. Multi-format Deliberative RAG 接口层实验

这里开始统一展示最新版 RAG 接口：

- `IndexedKnowledgeBase.acreate(...)`
- `KnowledgeAsset.from_path(...)`
- `RetrievalIntent.from_question(...)`
- `RagStrategyConfig.for_classic(...) / for_deliberative(...) / for_hybrid(...)`
- `deliberative_retrieve / search_knowledge_assets / open_retrieved_evidence`

默认推荐优先使用多格式、主动式检索链路，而不是只停留在最小 `chunk top-k` 示例。


In [ ]:
from agentorch import IndexedKnowledgeBase
from agentorch.knowledge import Document, RetrievalIntent

knowledge_base = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(id='doc-1', text='agentorch is a code-first, async-first agent orchestration framework for Python.'),
        Document(id='doc-2', text='Deliberative RAG retrieves evidence with source routing, structure-aware extraction, and coverage checks.'),
        Document(id='doc-3', text='Hybrid RAG can combine classic coarse recall with deliberative evidence refinement.'),
    ]
)

report = await knowledge_base.get_retriever().retrieve_report(
    RetrievalIntent.from_question(
        'agentorch framework and deliberative rag',
        must_cover=['agentorch', 'deliberative'],
        max_documents=4,
    )
)

print('--- summary ---')
print(report.summary)
print('--- coverage ---')
print(report.coverage.model_dump())
print('--- visited sources ---')
print(report.visited_sources)
print('--- citations ---')
for item in report.citations:
    print(item.model_dump())


## 13. 把检索接入 Runtime

这里推荐直接把 RAG 和上下文治理一起装进 runtime：

- 用 `rag=` 选择 classic / deliberative / hybrid
- 用 `RuntimeConfig.agent(..., context_policy=..., state_policy=..., memory_policy=...)` 显式装配
- 用 `ContextPolicy.evidence_friendly(...)` 控制 evidence、citation、report 是否进 prompt
- 运行后直接查看 `result.reasoning_metadata['resolved_policies']`


In [ ]:
from agentorch import ContextPolicy, IndexedKnowledgeBase, MemoryPolicy, StatePolicy, create_agent
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig

knowledge_base = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(
            id='doc-1',
            text='agentorch is designed for code-first, async-first agent orchestration in Python.',
            metadata={'scopes': ['product']},
        ),
        Document(
            id='doc-2',
            text='The runtime can mount retrieved evidence into the current agent context and report uncovered items.',
            metadata={'scopes': ['product']},
        ),
    ]
)

agent = create_agent(
    model=make_notebook_model(),
    knowledge_base=knowledge_base,
    name='nb-rag-agent',
    runtime_config=RuntimeConfig.agent(
        context_policy=ContextPolicy.evidence_friendly(selection_mode='hybrid'),
        state_policy=StatePolicy(retention_mode='state_plus_memory'),
        memory_policy=MemoryPolicy.long_horizon(),
        rag=RagStrategyConfig.for_hybrid(
            knowledge_scope=['product'],
            must_cover=['agentorch', 'retrieved evidence'],
            max_steps=3,
        ),
        reasoning='react',
    ),
)

result = await agent.run(
    'What is agentorch designed for, and how does runtime retrieval work?',
    thread_id='nb-rag-001',
)

print(result.output_text)
print('facade:', agent.export_blueprint()['facade'])
print('resolved policies:', result.reasoning_metadata.get('resolved_policies'))
print('context budget:', result.reasoning_metadata.get('context_budget_report'))


## 14. AgentRegistry 实验

多智能体系统里，agent 需要先注册，再被 workflow 或 supervisor 调度。


In [ ]:
from agentorch import AgentRegistry, AgentSpec

registry = AgentRegistry()
print(registry.list_specs())


## 15. 构建一个 specialist agent 并注册

下面构建一个最小 specialist agent，并注册到 `AgentRegistry`。


In [ ]:
from pydantic import BaseModel
from agentorch import AgentCapability, AgentRegistry, AgentSpec, ToolRegistry, create_agent, tool
from agentorch.config import RuntimeConfig


class EchoInput(BaseModel):
    text: str


@tool(description='Echo a message as structured data.')
async def echo(input: EchoInput):
    return {'echo': input.text}


def build_specialist_agent(description: str):
    return create_agent(
        model=make_notebook_model(),
        tools=ToolRegistry.from_tools(echo),
        runtime_config=RuntimeConfig.agent(
            system_prompt=description,
            reasoning='react',
        ),
    )


registry = AgentRegistry()
planner_agent = build_specialist_agent('You are a planning specialist for decomposition tasks.')
registry.register(
    AgentSpec.assistant(
        'planner',
        description='Planning specialist for decomposition tasks',
        capabilities=[AgentCapability.PLAN, AgentCapability.TOOL_USE],
        tools=['echo'],
        knowledge_scopes=['architecture'],
        preferred_reasoning_kind='plan_execute',
    ),
    planner_agent,
)

print(registry.list_specs())


## 16. Supervisor 多智能体动态委派实验

如果你只是想快速搭一个 supervisor 团队，优先用 `create_multi_agent(...)`。

上一节保留 `AgentRegistry` 是为了让你直接看 spec 和注册信息；这一节切回稳定公开面来运行一个最小多智能体委派示例。


In [ ]:
from agentorch import AgentCapability, create_multi_agent

agent = create_multi_agent(
    model=make_notebook_model(),
    agents=[
        {
            'agent': planner_agent,
            'name': 'planner',
            'role': 'planner',
            'description': 'Planning specialist for decomposition tasks',
            'capabilities': [AgentCapability.PLAN, AgentCapability.TOOL_USE],
            'knowledge_scope': ['architecture'],
        }
    ],
    system_prompt='You supervise specialists and route planning tasks to the best agent.',
    name='nb-supervisor-team',
)

result = await agent.run(
    'Please plan the implementation steps for a Python agent framework.',
    thread_id='nb-supervisor-001',
)

print(result.output_text)
print('facade:', agent.export_blueprint()['facade'])


## 17. Workflow 中的 agent 节点实验

这条链路演示在稳定多智能体公开面上挂 `workflow`，并通过 `agent` 节点完成委派。


In [ ]:
from agentorch import AgentCapability, Workflow, create_multi_agent
from agentorch.config import RuntimeConfig
from agentorch.workflow import Node

workflow = Workflow.chain(
    Node.agent(
        'delegate',
        'planner',
        goal='Please plan the steps for building an async Python agent runtime.',
        output_key='planner_output',
    )
)

agent = create_multi_agent(
    model=make_notebook_model(),
    agents=[
        {
            'agent': planner_agent,
            'name': 'planner',
            'role': 'planner',
            'description': 'Planning specialist for decomposition tasks',
            'capabilities': [AgentCapability.PLAN, AgentCapability.TOOL_USE],
            'knowledge_scope': ['architecture'],
        }
    ],
    workflow=workflow,
    system_prompt='You coordinate specialists for workflow nodes.',
    runtime_config=RuntimeConfig.workflow(reasoning='react'),
    name='nb-workflow-agent-team',
)

result = await agent.run('ignored by workflow node config', thread_id='nb-workflow-agent-001')
print(result.output_text)
print('facade:', agent.export_blueprint()['facade'])


## 18. 当前完整装配模板

这一节展示当前版本比较完整的装配方式：

- `ContextPolicy + StatePolicy + CoordinationPolicy + MemoryPolicy`
- `reasoning + rag`
- `tools + sandbox`
- `knowledge_base`
- `agent_registry + supervisor`

也就是你现在做研究型智能体或多智能体编排时最接近真实项目的一种装配方式。


In [ ]:
import json
from pathlib import Path
from pprint import pprint

from agentorch import Agent, ContextPolicy, IndexedKnowledgeBase, SandboxManager, ToolRegistry
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig
from agentorch.sandbox import SandboxPolicy


# -----------------------------
# 1. 创建沙箱
# -----------------------------
sandbox = SandboxManager(
    policy=SandboxPolicy(
        allowed_paths=[Path.cwd()],
        command_allowlist=["python", "git", "powershell", "cmd"],
        timeout=15.0,
    )
)

# -----------------------------
# 2. 创建工具注册表
# -----------------------------
tools = ToolRegistry.with_bundles(
    workspace_root=Path.cwd(),
    sandbox=sandbox,
)

# -----------------------------
# 3. 创建知识库
# -----------------------------
knowledge_base = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(
            id="doc-1",
            text="agentorch supports runtime, tools, memory, workflows, reasoning, and multi-agent orchestration.",
            metadata={"scopes": ["overview"]},
        ),
        Document(
            id="doc-2",
            text="Deliberative RAG can route sources and validate coverage before returning evidence.",
            metadata={"scopes": ["overview", "rag"]},
        ),
    ]
)

# -----------------------------
# 4. 创建运行配置
# -----------------------------
# 为了更容易观察工具调用过程，这里用相对轻一点的配置
runtime_config = RuntimeConfig.agent(
    reasoning="react",
    context_policy=ContextPolicy.lean(),
    rag=RagStrategyConfig.for_deliberative(
        knowledge_scope=["overview"],
        max_steps=1,
    ),
    max_steps=4,
)

# -----------------------------
# 5. 创建 Agent
# -----------------------------
agent = await Agent.acreate(
    model=make_notebook_model(),
    tools=tools,
    sandbox=sandbox,
    knowledge_base=knowledge_base,
    config=runtime_config,
)

# -----------------------------
# 6. 给底层模型 generate 打补丁
#    打印每轮模型 token 消耗
# -----------------------------
call_logs = []
original_generate = agent.runtime.model.generate

async def traced_generate(request):
    call_index = len(call_logs) + 1

    response = await original_generate(request)

    usage = response.usage
    item = {
        "call_index": call_index,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "finish_reason": response.finish_reason,
        "tool_calls": len(response.tool_calls),
        "content_preview": (response.content or "")[:200],
    }
    call_logs.append(item)

    print(f"\n========== 模型调用 #{call_index} ==========")
    print(f"输入 token: {usage.prompt_tokens}")
    print(f"输出 token: {usage.completion_tokens}")
    print(f"总 token: {usage.total_tokens}")
    print(f"finish_reason: {response.finish_reason}")
    print(f"tool_calls 数量: {len(response.tool_calls)}")

    if response.tool_calls:
        print("工具调用计划:")
        for tc in response.tool_calls:
            print(f"  tool_name: {tc.name}")
            print(f"  tool_call_id: {tc.id}")
            print(f"  arguments:")
            pprint(tc.arguments)

    if response.content:
        print("模型输出预览:")
        print(response.content[:300])

    return response

agent.runtime.model.generate = traced_generate


# -----------------------------
# 7. 流式运行，打印全过程事件
# -----------------------------
final_result = None

async for event in agent.run(
    "请先用一句话介绍你自己，然后查看当前目录有哪些文件，并简要列出来。",
    thread_id="tool-trace-1",
    stream=True,
):
    print(f"\n==================== 事件: {event.event_type} ====================")

    # 你最关心的是 payload
    if event.payload:
        pprint(event.payload)

    # 如果事件里带有工具调用信息
    if event.tool_calls:
        print("tool_calls:")
        for tc in event.tool_calls:
            print(f"  tool_name: {tc.name}")
            print(f"  tool_call_id: {tc.id}")
            print(f"  arguments:")
            pprint(tc.arguments)

    # 流式文本增量
    if event.delta_text:
        print("delta_text:")
        print(event.delta_text)

    # 最终结果事件
    if event.event_type == "final_result" and event.result is not None:
        final_result = event.result


# -----------------------------
# 8. 打印最终结果
# -----------------------------
print("\n\n========== 最终输出 ==========")
if final_result is not None:
    print(final_result.output_text)

    print("\n========== 最终总 token ==========")
    print(f"总输入 token: {final_result.usage.prompt_tokens}")
    print(f"总输出 token: {final_result.usage.completion_tokens}")
    print(f"总 token: {final_result.usage.total_tokens}")

    print("\n========== 最终工具结果 ==========")
    for i, tr in enumerate(final_result.tool_results, start=1):
        print(f"\n工具 #{i}")
        print(f"tool_name: {tr.tool_name}")
        print(f"is_error: {tr.is_error}")
        print(f"error_message: {tr.error_message}")
        print(f"duration: {tr.duration}")
        print("output:")
        pprint(tr.output)


# -----------------------------
# 9. 打印逐轮模型调用统计
# -----------------------------
print("\n========== 每轮模型调用 token 明细 ==========")
for item in call_logs:
    print(
        f"第 {item['call_index']} 轮 | "
        f"输入: {item['prompt_tokens']} | "
        f"输出: {item['completion_tokens']} | "
        f"总计: {item['total_tokens']} | "
        f"finish_reason: {item['finish_reason']} | "
        f"tool_calls: {item['tool_calls']}"
    )

# -----------------------------
# 10. 关闭资源
# -----------------------------
await agent.aclose()


In [ ]:
import json

log_path = "tool_trace_log.jsonl"

final_result = None

with open(log_path, "w", encoding="utf-8") as f:
    async for event in agent.run(
        "请先用一句话介绍你自己，然后查看当前目录有哪些文件，并简要列出来。",
        thread_id="tool-trace-file",
        stream=True,
    ):
        record = {
            "event_type": event.event_type,
            "payload": event.payload,
            "delta_text": event.delta_text,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

        if event.event_type == "final_result" and event.result is not None:
            final_result = event.result

print("日志已保存到:", log_path)


## 19. 推理框架总览

当前版本已经支持通过统一工厂 API 创建五种推理框架：

- `CoT`：线性分步推理
- `ReAct`：推理与工具行动交替
- `Plan-and-Execute`：先规划再执行
- `ToT`：多分支候选探索与剪枝
- `Reflexion`：尝试、反思、再尝试

这些框架既可以通过 `create_reasoning_framework(...)` 单独创建和检查，也可以通过 `RuntimeConfig.agent(reasoning=...)` / `RuntimeConfig.workflow(reasoning=...)` 走统一入口。


In [ ]:
from agentorch import ReasoningStrategyConfig, create_reasoning_framework

strategy_configs = [
    ReasoningStrategyConfig.cot(config={'max_steps': 6}),
    ReasoningStrategyConfig.react(config={'max_steps': 8}),
    ReasoningStrategyConfig.plan_execute(config={'max_planning_steps': 5, 'max_execution_steps': 8}),
    ReasoningStrategyConfig.tot(config={'branch_factor': 3, 'max_depth': 3, 'top_k': 2}),
    ReasoningStrategyConfig.reflexion(config={'max_attempts': 3, 'enable_self_reflection': True}),
]

for strategy in strategy_configs:
    framework = create_reasoning_framework(strategy.kind, **strategy.config)
    print(strategy.model_dump())
    print(framework.config)
    print('-' * 80)



## 20. CoT 实验

适合做结构化解释、数学推导、设计拆解等不强依赖工具交互的任务。

运行后可以同时查看：

- `result.output_text`
- `result.reasoning_kind`
- `result.reasoning_trace`


In [ ]:
from agentorch import Agent
from agentorch.config import RuntimeConfig

agent = await Agent.acreate(
    model=make_notebook_model(),
    config=RuntimeConfig.agent(
        reasoning='cot',
        system_prompt='你是一个底层架构分析师。',
    ),
)

result = await agent.run(
    '请分步骤分析，一个底层智能体编排框架为什么要有 model、runtime、tools、memory、workflow 这几个层。',
    thread_id='nb-reasoning-cot-001',
)

print('reasoning kind:', result.reasoning_kind)
print('--- output ---')
print(result.output_text)
print('--- reasoning trace ---')
print(result.reasoning_trace)



## 21. Plan-and-Execute / ToT / Reflexion 快速实验

这三个更适合研究型编排：

- `Plan-and-Execute`：任务链长、流程明确
- `ToT`：需要比较多个候选方案
- `Reflexion`：希望 agent 先尝试，再自我改进


In [ ]:
from agentorch import Agent
from agentorch.config import RuntimeConfig

frameworks = [
    ('plan_execute', '请先规划，再给出一个 Python 智能体系统的实现步骤。'),
    ('tot', '请比较三种多智能体协作架构，并择优给出推荐。'),
    ('reflexion', '请先给出一个方案，再自我检查并改进它。'),
]

for name, prompt in frameworks:
    agent = await Agent.acreate(
        model=make_notebook_model(),
        config=RuntimeConfig.agent(reasoning=name),
    )
    result = await agent.run(prompt, thread_id=f'nb-{name}-001')
    print('=' * 80)
    print('framework:', name)
    print('output:')
    print(result.output_text)
    print('trace:')
    print(result.reasoning_trace)



## 22. 架构设计案例一：单 Agent 架构分析师

这个案例适合你做“架构顾问型 agent”：

- 用 `Plan-and-Execute` 做设计拆解
- 用 `RAG` 注入你的设计规范或技术文档
- 用 `ContextPolicy.evidence_friendly()` + `MemoryPolicy.long_horizon()` 打开更重的上下文与记忆治理
- 输出最终方案 + 推理过程 + 已解析策略


In [ ]:
from agentorch import ContextPolicy, IndexedKnowledgeBase, MemoryPolicy, create_agent
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig

design_kb = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(id='arch-1', text='底层智能体框架应分离模型层、工具层、推理层、运行时层与观测层。', metadata={'scopes': ['architecture']}),
        Document(id='arch-2', text='多智能体系统建议优先使用结构化任务包与共享工件，而不是无限自由对话。', metadata={'scopes': ['architecture', 'multi-agent']}),
    ]
)

agent = create_agent(
    model=make_notebook_model(),
    knowledge_base=design_kb,
    name='nb-arch-analyst',
    runtime_config=RuntimeConfig.agent(
        reasoning='plan_execute',
        context_policy=ContextPolicy.evidence_friendly(),
        memory_policy=MemoryPolicy.long_horizon(),
        rag=RagStrategyConfig.for_deliberative(
            knowledge_scope=['architecture'],
            must_cover=['模型层', '运行时层'],
            max_steps=3,
        ),
    ),
)

result = await agent.run(
    '请基于已有知识，设计一个单 Agent 架构分析师的系统骨架，并说明为什么这样分层。',
    thread_id='nb-arch-single-001',
)

print(result.output_text)
print('facade:', agent.export_blueprint()['facade'])


## 23. 架构设计案例二：RAG + 多智能体架构设计

这一节把“不同 specialist 绑定不同推理框架”落成一个最小案例：

- `planner`：`Plan-and-Execute`
- `researcher`：`ReAct`
- `reviewer`：`Reflexion`
- `create_multi_agent(...)`：统一装配 supervisor 团队
- `CoordinationPolicy.route_mode`：如果后面要做象群实验，可再切到 `experiments.elephant_context` 插件装配

为了让示例更稳定，这里显式给每个 specialist 写出 `name / description / capabilities / knowledge_scope`，实际路由仍由 supervisor 决定，不再假设固定顺序。


In [ ]:
from agentorch import AgentCapability, IndexedKnowledgeBase, create_agent, create_multi_agent
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document, RagStrategyConfig

shared_kb = await IndexedKnowledgeBase.acreate(
    documents=[
        Document(id='arch-1', text='底层智能体框架应分离模型层、工具层、推理层、运行时层与观测层。', metadata={'scopes': ['architecture']}),
        Document(id='arch-2', text='多智能体系统建议优先使用结构化任务包、共享工件与 supervisor 路由。', metadata={'scopes': ['architecture', 'multi-agent']}),
        Document(id='arch-3', text='Hybrid RAG can combine classic coarse recall with deliberative evidence extraction.', metadata={'scopes': ['architecture', 'rag']}),
    ]
)

planner = create_agent(
    model=make_notebook_model(),
    knowledge_base=shared_kb,
    name='planner',
    description='Architecture planning specialist',
    runtime_config=RuntimeConfig.agent(
        reasoning='plan_execute',
        rag=RagStrategyConfig.for_hybrid(knowledge_scope=['architecture']),
    ),
)

researcher = create_agent(
    model=make_notebook_model(),
    knowledge_base=shared_kb,
    name='researcher',
    description='Architecture evidence gathering specialist',
    runtime_config=RuntimeConfig.agent(
        reasoning='react',
        rag=RagStrategyConfig.for_deliberative(knowledge_scope=['architecture', 'rag']),
    ),
)

reviewer = create_agent(
    model=make_notebook_model(),
    knowledge_base=shared_kb,
    name='reviewer',
    description='Architecture review specialist',
    runtime_config=RuntimeConfig.agent(
        reasoning='reflexion',
        rag=RagStrategyConfig.for_deliberative(knowledge_scope=['architecture', 'rag']),
    ),
)

orchestrator = create_multi_agent(
    model=make_notebook_model(),
    agents=[
        {
            'agent': planner,
            'name': 'planner',
            'role': 'planner',
            'description': 'Architecture planning specialist',
            'capabilities': [AgentCapability.PLAN],
            'knowledge_scope': ['architecture'],
        },
        {
            'agent': researcher,
            'name': 'researcher',
            'role': 'researcher',
            'description': 'Architecture evidence gathering specialist',
            'capabilities': [AgentCapability.RETRIEVE],
            'knowledge_scope': ['architecture', 'rag'],
        },
        {
            'agent': reviewer,
            'name': 'reviewer',
            'role': 'reviewer',
            'description': 'Architecture review specialist',
            'capabilities': [AgentCapability.REVIEW],
            'knowledge_scope': ['architecture', 'rag'],
        },
    ],
    system_prompt='You coordinate planner, researcher, and reviewer, then synthesize a final architecture proposal.',
    name='nb-arch-multi-team',
    reasoning='react',
)

result = await orchestrator.run(
    '请给出一个 RAG + 多智能体 的底层架构设计，并指出 planner、researcher 与 reviewer 如何分工。',
    thread_id='nb-arch-multi-001',
)

print(result.output_text)
print('facade:', orchestrator.export_blueprint()['facade'])
print('members:', [member['name'] for member in orchestrator.export_blueprint()['members']])


## 24. Generic Composition Boundary (v1)

AgentTorch v1 no longer ships research-specific preset agents in the main package.

Use the generic stable surface instead:

- `create_agent(...)` / `create_multi_agent(...)`
- `AgentDesign` / `RoleDesign` / `TeamDesign` + `compose_agent(...)` / `compose_team(...)`
- `create_agent_evolution(...)` / `create_multi_agent_evolution(...)`
- explicit `RuntimeConfig`, `ToolRegistry`, memory, knowledge, reasoning, and extension plugins
- experiment-specific mechanisms such as elephant context or nutcracker memory should stay as composable modules, not top-level preset agents

如果你想看 facade-first 示例和 lower-level runtime 示例如何分层，也请同步参考仓库里的 `examples/README.md`。


In [ ]:
import agentorch
from agentorch import AgentCapability, AgentDesign, RoleDesign, TeamDesign, compose_agent, compose_team, create_agent, create_multi_agent
from agentorch.core import Message, ModelRequest, ModelResponse, UsageInfo
from agentorch.models.base import BaseModelAdapter


class NotebookEchoModel(BaseModelAdapter):
    def __init__(self, reply: str) -> None:
        self.reply = reply

    async def generate(self, request: ModelRequest) -> ModelResponse:
        return ModelResponse(
            message=Message(role='assistant', content=self.reply),
            content=self.reply,
            finish_reason='stop',
            usage=UsageInfo(total_tokens=1),
        )


facade_agent = create_agent(
    model=NotebookEchoModel('facade-ready'),
    name='facade-solo',
    system_prompt='You are a facade-built solo agent.',
    reasoning='react',
)

design_agent = compose_agent(
    AgentDesign.named('design-solo', model=NotebookEchoModel('design-ready'), profile='default')
    .with_reasoning('react')
    .with_runtime_config(system_prompt='You are a design-built solo agent.')
)

facade_team_member = create_agent(
    model=NotebookEchoModel('planner-ready'),
    name='facade-planner',
    system_prompt='You are a planner.',
    reasoning='plan_execute',
)

facade_team = create_multi_agent(
    model=NotebookEchoModel('coordinator-ready'),
    agents=[
        {
            'agent': facade_team_member,
            'name': 'planner',
            'role': 'planner',
            'description': 'Facade-built planner',
            'capabilities': [AgentCapability.PLAN],
        }
    ],
    system_prompt='Route tasks to the planner.',
    name='facade-team',
)

design_team = compose_team(
    TeamDesign(
        name='design-team',
        system_prompt='Route tasks to the reviewer.',
        roles=[
            RoleDesign(
                name='reviewer',
                description='Design-built reviewer',
                capabilities=[AgentCapability.REVIEW],
                design=AgentDesign(model=NotebookEchoModel('review-ready')).with_reasoning('reflexion'),
            )
        ],
    )
)

try:
    print('agentorch file:', agentorch.__file__)
    print('stable facade entrypoints:', hasattr(agentorch, 'create_agent'), hasattr(agentorch, 'create_multi_agent'))
    print('design entrypoints:', hasattr(agentorch, 'AgentDesign'), hasattr(agentorch, 'compose_team'))
    print('evolution entrypoints:', hasattr(agentorch, 'create_agent_evolution'), hasattr(agentorch, 'create_multi_agent_evolution'))
    print('facade agent facade ->', facade_agent.export_blueprint()['facade'])
    print('design agent facade ->', design_agent.export_blueprint()['facade'])
    print('facade team facade ->', facade_team.export_blueprint()['facade'])
    print('design team facade ->', design_team.export_blueprint()['facade'])
    print('facade team members ->', [m['name'] for m in facade_team.export_blueprint()['members']])
    print('design team members ->', [m['name'] for m in design_team.export_blueprint()['members']])
    print('facade agent run ->', (await facade_agent.run('say hi', thread_id='nb-v1-facade-agent')).output_text)
    print('design agent run ->', (await design_agent.run('say hi', thread_id='nb-v1-design-agent')).output_text)
    print('facade team run ->', (await facade_team.run('plan this', thread_id='nb-v1-facade-team')).output_text)
    print('design team run ->', (await design_team.run('review this', thread_id='nb-v1-design-team')).output_text)
finally:
    await facade_agent.aclose()
    await design_agent.aclose()
    await facade_team_member.aclose()
    await facade_team.aclose()
    await design_team.aclose()


## 25. 设计建议

如果你要继续把这个框架往研究或工程底盘推进，当前更值得优先增强的是：

- 把 notebook / README / examples 全部统一到 `policy + plugin` 写法
- 继续扩展多格式 RAG ingestion 与 evidence opening 能力
- 将场景模板沉到 examples 或外部资产，而不是重新放回主包公开面
- 把用户自定义 strategy / profile 注册做成更稳定的企业级接口
- 增强 supervisor 的 agent 选择策略与协作拓扑可视化
- 增强 observability，把 `resolved_policies / context_budget / coordination_report` 做成更好读的报告


## 26. 实验排错清单

如果再次遇到问题，优先按这个顺序检查：

1. notebook 里是否误用了 `run_sync()`
2. 是否修改过本地代码但没有重启 kernel
3. 是否复用了旧的 `thread_id`
4. `.env` 是否被正确读取
5. `base_url` 是否被规整成 `/v1`
6. sandbox 的 `allowed_paths` 和 `command_allowlist` 是否允许当前执行
7. 多 agent 实验前是否先把 specialist agent 注册到 `AgentRegistry`
8. RAG 实验是否通过 `RuntimeConfig.agent(...)` 或 `RuntimeConfig.workflow(...)` 正确装入 `rag=`
9. 是否显式配置了 core policy，但又被 node / metadata override 覆盖
10. 如果跑 pytest，是否设置了 `PYTEST_DISABLE_PLUGIN_AUTOLOAD=1` 来避免本机全局插件污染

最常见的几条仍然是：

- notebook 要用 `await agent.run(...)`
- tool / multi-agent 调试失败时，请换新的 `thread_id` 或先 `clear_thread()`
- 遇到上下文注入和预期不一致时，先看 `result.reasoning_metadata['resolved_policies']`


## 27. 联网搜索工具 API 参数测试

这一节专门验证三件事：

- `Runtime.acreate(..., include_web_tools=True, web_search_api_key=...)` 这条低层 runtime 参数路径是否可用
- `create_agent(..., tool_bundles={'include_web': True, 'brave_api_key': ...})` 这条 facade 路径是否可用，同时还能保留用户自定义工具
- 单个智能体是否可以直接通过 `brave_search` 完成一次网页搜索调用

说明：

- 这一节复用第 2 节已经加载到环境中的变量，不再重复手写 `.env` 解析逻辑
- 不会打印完整 key，只打印是否读取成功
- 这一节是“活测试”，会真正调用 Brave Search
- 如果当前网络环境无法连通 Brave，会输出诊断信息而不是直接让整个 cell 失败


In [ ]:
import sys
from pathlib import Path

from pydantic import BaseModel

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

brave_api_key = pick_env('BRAVE_API_KEY', 'BRAVE_SEARCH_API_KEY')
print('BRAVE_API_KEY loaded:', bool(brave_api_key))
assert brave_api_key, '未在当前环境变量或 notebook 顶部加载结果中找到 BRAVE_API_KEY'

# 清掉旧模块缓存，确保 notebook 读取的是当前最新本地代码。
for name in list(sys.modules):
    if name == 'agentorch' or name.startswith('agentorch.'):
        sys.modules.pop(name, None)

from agentorch import Runtime, ToolRegistry, create_agent, create_brave_search_tool, tool
from agentorch.core import Message, ModelRequest, ModelResponse, ToolCall, UsageInfo
from agentorch.knowledge import Document
from agentorch.models.base import BaseModelAdapter
from agentorch.tools import ToolError


class SimpleEchoModel(BaseModelAdapter):
    async def generate(self, request: ModelRequest) -> ModelResponse:
        return ModelResponse(
            message=Message(role='assistant', content='tooling test completed'),
            content='tooling test completed',
            finish_reason='stop',
            usage=UsageInfo(total_tokens=1),
        )


class SingleAgentWebSearchModel(BaseModelAdapter):
    def __init__(self) -> None:
        self.calls = 0

    async def generate(self, request: ModelRequest) -> ModelResponse:
        self.calls += 1
        if self.calls == 1:
            tool_call = ToolCall(
                id='call-brave-1',
                name='brave_search',
                arguments={
                    'query': 'multi-agent systems architecture',
                    'count': 2,
                    'search_lang': 'en',
                    'country': 'US',
                },
            )
            return ModelResponse(
                message=Message(role='assistant', content='Using brave_search to look up the web.', tool_calls=[tool_call]),
                content='Using brave_search to look up the web.',
                tool_calls=[tool_call],
                finish_reason='tool_calls',
                usage=UsageInfo(total_tokens=1),
            )

        latest_tool_message = next((m.content for m in reversed(request.messages) if m.role == 'tool'), '')
        return ModelResponse(
            message=Message(role='assistant', content=f'Single-agent web search completed. Tool payload summary: {latest_tool_message[:400]}'),
            content=f'Single-agent web search completed. Tool payload summary: {latest_tool_message[:400]}',
            finish_reason='stop',
            usage=UsageInfo(total_tokens=1),
        )


class EchoInput(BaseModel):
    text: str


@tool(description='Return a tagged echo for notebook testing.')
async def notebook_echo(input: EchoInput):
    return {'echo': input.text, 'source': 'custom_tool'}


async def safe_tool_execute(registry: ToolRegistry, name: str, arguments: dict) -> dict:
    try:
        result = await registry.execute(name, arguments)
        return {'ok': True, 'data': result.data}
    except ToolError as exc:
        return {'ok': False, 'error': str(exc)}


def print_search_summary(title: str, payload: dict) -> None:
    print()
    print(title)
    if not payload.get('ok'):
        print('search failed:', payload.get('error'))
        return
    results = payload.get('data', {}).get('results', [])
    print('result count:', len(results))
    for item in results[:3]:
        print('-', item.get('title'))
        print('  ', item.get('url'))


print()
print('[1] 直接测试 brave_search 工具')
brave_registry = ToolRegistry.empty()
brave_registry.register(create_brave_search_tool(api_key=brave_api_key, timeout=60.0))
brave_payload = await safe_tool_execute(
    brave_registry,
    'brave_search',
    {
        'query': 'multi-agent systems architecture',
        'count': 3,
        'search_lang': 'en',
        'country': 'US',
    },
)
print_search_summary('[1] brave_search direct call', brave_payload)


print()
print('[2] Runtime.acreate 低层 API 参数测试')
runtime = await Runtime.acreate(
    model=SimpleEchoModel(),
    include_web_tools=True,
    web_search_api_key=brave_api_key,
    custom_tools=[notebook_echo],
    knowledge_documents=[
        Document(
            id='runtime-web-1',
            text='Runtime should accept web tools and custom tools through API parameters.',
            metadata={'scopes': ['web']},
        )
    ],
)
try:
    print('runtime tools contain brave_search:', 'brave_search' in runtime.tools)
    print('runtime tools contain notebook_echo:', 'notebook_echo' in runtime.tools)
    assert 'brave_search' in runtime.tools
    assert 'notebook_echo' in runtime.tools

    runtime_payload = await safe_tool_execute(
        runtime.tools,
        'brave_search',
        {
            'query': 'agent orchestration supervisor worker',
            'count': 2,
            'search_lang': 'en',
            'country': 'US',
        },
    )
    print_search_summary('[2] runtime brave_search', runtime_payload)
finally:
    await runtime.aclose()


print()
print('[3] create_agent facade + web bundle 测试')
agent = create_agent(
    model=SimpleEchoModel(),
    tools=[notebook_echo],
    tool_bundles={
        'include_filesystem': False,
        'include_execution': False,
        'include_git': False,
        'include_web': True,
        'brave_api_key': brave_api_key,
    },
    system_prompt='Tooling test agent.',
)
try:
    print('agent tools contain brave_search:', 'brave_search' in agent.runtime.tools)
    print('agent tools contain notebook_echo:', 'notebook_echo' in agent.runtime.tools)
    assert 'brave_search' in agent.runtime.tools
    assert 'notebook_echo' in agent.runtime.tools

    agent_payload = await safe_tool_execute(
        agent.runtime.tools,
        'brave_search',
        {
            'query': 'planner executor critic multi-agent',
            'count': 2,
            'search_lang': 'en',
            'country': 'US',
        },
    )
    print_search_summary('[3] facade brave_search', agent_payload)

    custom_result = await agent.runtime.tools.execute('notebook_echo', {'text': 'web + custom tools ok'})
    print('custom tool result:', custom_result.data)
    assert custom_result.data['source'] == 'custom_tool'
finally:
    await agent.aclose()


print()
print('[4] facade 单个智能体直接调用网页搜索')
single_agent = create_agent(
    model=SingleAgentWebSearchModel(),
    tool_bundles={
        'include_filesystem': False,
        'include_execution': False,
        'include_git': False,
        'include_web': True,
        'brave_api_key': brave_api_key,
    },
    system_prompt='Use brave_search when helpful.',
)
try:
    single_agent_result = await single_agent.run(
        'Please search the web for multi-agent architecture examples.',
        thread_id='single-agent-web-search-demo',
    )
    print(single_agent_result.output_text)
except Exception as exc:
    print('single-agent web search failed:', type(exc).__name__, str(exc))
finally:
    await single_agent.aclose()


print()
print('实验结束。如果上面出现 connect timeout，一般不是 API 参数接入问题，而是当前环境到 Brave 的网络连通性问题。')


## 28. 多媒体模型示例（复用顶部 `.env`）

这一节单独把三类媒体能力再集中展示一遍，但配置仍然完全复用第 2 节和 3.1 节已经解析好的 `.env` 常量。

覆盖内容：

- 模型级 `synthesize_speech(...)`
- 模型级 `generate_image(...)`
- 模型级 `analyze_video(...)`
- `include_media=True` 注入 agent 后的工具列表

说明：真实媒体请求默认仍然关闭，先看代码结构；需要时再把开关改成 `True`。


In [ ]:
media_model = make_notebook_model()
try:
    print('chat model   ->', NOTEBOOK_CHAT_MODEL)
    print('speech model ->', NOTEBOOK_SPEECH_MODEL)
    print('image model  ->', NOTEBOOK_IMAGE_MODEL)
    print('video model  ->', NOTEBOOK_VIDEO_MODEL)

    media_registry = ToolRegistry.with_bundles(
        workspace_root=NOTEBOOK_PROJECT_ROOT,
        include_filesystem=False,
        include_execution=False,
        include_git=False,
        include_media=True,
        model=media_model,
    )
    print('media tools ->', [spec['function']['name'] for spec in media_registry.list_specs()])
finally:
    await media_model.aclose()


In [ ]:
RUN_MEDIA_MODEL_EXAMPLES = False
MEDIA_NOTEBOOK_DIR = NOTEBOOK_PROJECT_ROOT / '.agentorch' / 'notebook_media_examples'
VIDEO_EXAMPLE_PATH = NOTEBOOK_VIDEO_DEMO_PATH


async def demo_media_model_methods(video_path: Path | None = None):
    model = make_notebook_model()
    try:
        MEDIA_NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

        speech_result = await model.synthesize_speech(
            '这是 notebook 里的文字转语音最小示例。',
            output_path=MEDIA_NOTEBOOK_DIR / 'demo_tts',
        )
        print('tts file   ->', speech_result.output_path)

        image_result = await model.generate_image(
            NOTEBOOK_IMAGE_GENERATION_PROMPT,
            output_path=MEDIA_NOTEBOOK_DIR / 'demo_image',
        )
        print('image file ->', image_result.output_path)

        if video_path is not None and video_path.exists():
            video_result = await model.analyze_video(
                prompt='请用三句话说明这个视频的主体内容，并指出最关键的动作。',
                video_path=video_path,
            )
            print('video analysis ->')
            print(video_result.content)
        else:
            print(f'skip video demo: file not found -> {video_path}')
    finally:
        await model.aclose()


if RUN_MEDIA_MODEL_EXAMPLES:
    await demo_media_model_methods(VIDEO_EXAMPLE_PATH)
else:
    print('Set RUN_MEDIA_MODEL_EXAMPLES = True before running this cell.')


In [ ]:
from agentorch import create_agent

media_agent = create_agent(
    model=make_notebook_model(),
    tool_bundles={
        'include_filesystem': False,
        'include_execution': False,
        'include_git': False,
        'include_media': True,
    },
    system_prompt='你是一个可以按需调用多媒体工具的 agent。',
)

print('agent media tools ->', media_agent.export_blueprint()['runtime']['tools'])
print('提示：如果你要让 agent 真正调用这些工具，请在下一格打开运行开关。')


In [ ]:
from uuid import uuid4

RUN_MEDIA_AGENT_TOOL_DEMO = False

if RUN_MEDIA_AGENT_TOOL_DEMO:
    media_thread_id = f'nb-media-agent-{uuid4().hex[:8]}'

    async for event in media_agent.run(
        '请调用 generate_image 工具，给我生成一个杨过和雕兄的写实风格照片，杨过和雕兄并肩作战，襄阳大战。',
        thread_id=media_thread_id,
        stream=True,
    ):
        if event.event_type == 'tool_called':
            for call in event.tool_calls:
                print(f'[进度] 正在调用工具: {call.name}')
        elif event.event_type == 'tool_result':
            print('[进度] 工具执行完成')
        elif event.event_type == 'final_result' and event.result is not None:
            print('[完成]')
            print(event.result.output_text)
else:
    print('Set RUN_MEDIA_AGENT_TOOL_DEMO = True before running this cell.')
